# Relatórios de Beneficiários ANS

Consultas Spark SQL sobre a modelagem dimensional da camada Gold.

In [ ]:
from pyspark.sql import SparkSession
import os

from utils import default_spark_local_dir

SPARK_LOCAL_DIR = default_spark_local_dir()
os.environ["SPARK_LOCAL_DIRS"] = SPARK_LOCAL_DIR
os.makedirs(SPARK_LOCAL_DIR, exist_ok=True)

spark = (
    SparkSession.builder
    .config("spark.local.dir", SPARK_LOCAL_DIR)
    .getOrCreate()
)

CATALOG = "spark_catalog"
GOLD_DB = "gold"

FATO_MOVIMENTO = f"{CATALOG}.{GOLD_DB}.fato_beneficiario_movimento"
DIM_OPERADORA = f"{CATALOG}.{GOLD_DB}.dim_operadora"
DIM_MUNICIPIO = f"{CATALOG}.{GOLD_DB}.dim_municipio"
DIM_PLANO = f"{CATALOG}.{GOLD_DB}.dim_plano"
DIM_PERFIL = f"{CATALOG}.{GOLD_DB}.dim_perfil_beneficiario"

def query(sql: str, limit: int | None = None):
    df = spark.sql(sql)
    if limit is not None:
        df = df.limit(limit)
    return df


## Competências Disponíveis

In [ ]:
query(f"""
SELECT
    id_cmpt_movel,
    ds_competencia,
    SUM(qt_beneficiario_ativo) AS beneficiarios_ativos,
    SUM(qt_beneficiario_aderido) AS beneficiarios_aderidos,
    SUM(qt_beneficiario_cancelado) AS beneficiarios_cancelados
FROM {FATO_MOVIMENTO}
GROUP BY id_cmpt_movel, ds_competencia
ORDER BY id_cmpt_movel DESC
""").show(50, truncate=False)


## Resumo da Última Competência

In [ ]:
query(f"""
WITH ultima_competencia AS (
    SELECT MAX(id_cmpt_movel) AS id_cmpt_movel
    FROM {FATO_MOVIMENTO}
)
SELECT
    f.id_cmpt_movel,
    f.ds_competencia,
    COUNT(*) AS linhas_fato,
    SUM(f.qt_beneficiario_ativo) AS beneficiarios_ativos,
    SUM(f.qt_beneficiario_aderido) AS beneficiarios_aderidos,
    SUM(f.qt_beneficiario_cancelado) AS beneficiarios_cancelados,
    SUM(f.qt_beneficiario_aderido) - SUM(f.qt_beneficiario_cancelado) AS saldo_movimento
FROM {FATO_MOVIMENTO} f
JOIN ultima_competencia u
    ON f.id_cmpt_movel = u.id_cmpt_movel
GROUP BY f.id_cmpt_movel, f.ds_competencia
""").show(truncate=False)


## Beneficiários Ativos Por UF

In [ ]:
query(f"""
WITH ultima_competencia AS (
    SELECT MAX(id_cmpt_movel) AS id_cmpt_movel
    FROM {FATO_MOVIMENTO}
)
SELECT
    m.sg_uf,
    SUM(f.qt_beneficiario_ativo) AS beneficiarios_ativos,
    SUM(f.qt_beneficiario_aderido) AS beneficiarios_aderidos,
    SUM(f.qt_beneficiario_cancelado) AS beneficiarios_cancelados
FROM {FATO_MOVIMENTO} f
JOIN ultima_competencia u
    ON f.id_cmpt_movel = u.id_cmpt_movel
JOIN {DIM_MUNICIPIO} m
    ON f.sk_municipio = m.sk_municipio
GROUP BY m.sg_uf
ORDER BY beneficiarios_ativos DESC
""").show(100, truncate=False)


## Top Operadoras Por Beneficiários Ativos

In [ ]:
query(f"""
WITH ultima_competencia AS (
    SELECT MAX(id_cmpt_movel) AS id_cmpt_movel
    FROM {FATO_MOVIMENTO}
)
SELECT
    o.cd_operadora,
    o.nm_razao_social,
    o.modalidade_operadora,
    SUM(f.qt_beneficiario_ativo) AS beneficiarios_ativos,
    SUM(f.qt_beneficiario_aderido) AS beneficiarios_aderidos,
    SUM(f.qt_beneficiario_cancelado) AS beneficiarios_cancelados
FROM {FATO_MOVIMENTO} f
JOIN ultima_competencia u
    ON f.id_cmpt_movel = u.id_cmpt_movel
JOIN {DIM_OPERADORA} o
    ON f.sk_operadora = o.sk_operadora
GROUP BY o.cd_operadora, o.nm_razao_social, o.modalidade_operadora
ORDER BY beneficiarios_ativos DESC
LIMIT 20
""").show(20, truncate=False)


## Perfil Por Sexo E Faixa Etária

In [ ]:
query(f"""
WITH ultima_competencia AS (
    SELECT MAX(id_cmpt_movel) AS id_cmpt_movel
    FROM {FATO_MOVIMENTO}
)
SELECT
    p.tp_sexo,
    p.de_faixa_etaria,
    SUM(f.qt_beneficiario_ativo) AS beneficiarios_ativos
FROM {FATO_MOVIMENTO} f
JOIN ultima_competencia u
    ON f.id_cmpt_movel = u.id_cmpt_movel
JOIN {DIM_PERFIL} p
    ON f.sk_perfil_beneficiario = p.sk_perfil_beneficiario
GROUP BY p.tp_sexo, p.de_faixa_etaria
ORDER BY p.tp_sexo, p.de_faixa_etaria
""").show(200, truncate=False)


## Beneficiários Por Tipo De Vínculo

In [ ]:
query(f"""
WITH ultima_competencia AS (
    SELECT MAX(id_cmpt_movel) AS id_cmpt_movel
    FROM {FATO_MOVIMENTO}
)
SELECT
    p.tipo_vinculo,
    SUM(f.qt_beneficiario_ativo) AS beneficiarios_ativos,
    SUM(f.qt_beneficiario_aderido) AS beneficiarios_aderidos,
    SUM(f.qt_beneficiario_cancelado) AS beneficiarios_cancelados
FROM {FATO_MOVIMENTO} f
JOIN ultima_competencia u
    ON f.id_cmpt_movel = u.id_cmpt_movel
JOIN {DIM_PERFIL} p
    ON f.sk_perfil_beneficiario = p.sk_perfil_beneficiario
GROUP BY p.tipo_vinculo
ORDER BY beneficiarios_ativos DESC
""").show(truncate=False)


## Evolução Mensal De Ativos, Adesões E Cancelamentos

In [ ]:
query(f"""
SELECT
    id_cmpt_movel,
    ds_competencia,
    SUM(qt_beneficiario_ativo) AS beneficiarios_ativos,
    SUM(qt_beneficiario_aderido) AS beneficiarios_aderidos,
    SUM(qt_beneficiario_cancelado) AS beneficiarios_cancelados,
    SUM(qt_beneficiario_aderido) - SUM(qt_beneficiario_cancelado) AS saldo_movimento
FROM {FATO_MOVIMENTO}
GROUP BY id_cmpt_movel, ds_competencia
ORDER BY id_cmpt_movel
""").show(500, truncate=False)


## Distribuição Por Contratação E Segmentação Do Plano

In [ ]:
query(f"""
WITH ultima_competencia AS (
    SELECT MAX(id_cmpt_movel) AS id_cmpt_movel
    FROM {FATO_MOVIMENTO}
)
SELECT
    pl.de_contratacao_plano,
    pl.de_segmentacao_plano,
    SUM(f.qt_beneficiario_ativo) AS beneficiarios_ativos
FROM {FATO_MOVIMENTO} f
JOIN ultima_competencia u
    ON f.id_cmpt_movel = u.id_cmpt_movel
JOIN {DIM_PLANO} pl
    ON f.sk_plano = pl.sk_plano
GROUP BY pl.de_contratacao_plano, pl.de_segmentacao_plano
ORDER BY beneficiarios_ativos DESC
""").show(100, truncate=False)


## Top Municípios Por Beneficiários Ativos

In [ ]:
query(f"""
WITH ultima_competencia AS (
    SELECT MAX(id_cmpt_movel) AS id_cmpt_movel
    FROM {FATO_MOVIMENTO}
)
SELECT
    m.sg_uf,
    m.nm_municipio,
    SUM(f.qt_beneficiario_ativo) AS beneficiarios_ativos
FROM {FATO_MOVIMENTO} f
JOIN ultima_competencia u
    ON f.id_cmpt_movel = u.id_cmpt_movel
JOIN {DIM_MUNICIPIO} m
    ON f.sk_municipio = m.sk_municipio
GROUP BY m.sg_uf, m.nm_municipio
ORDER BY beneficiarios_ativos DESC
LIMIT 50
""").show(50, truncate=False)
